# The fundamentals

Many problems in computer graphics can be written as optimization problems. We are looking for a configuration of variables $x$ (vertex positions, weights, distances, correspondences, ...) that makes some objective $f(x)$ as small or as large as possible, possibly subject to constraints,
$$ \min_x f(x) \quad \text{subject to} \quad x \in \Pi. $$
Writing the problem this way is half the battle. The other half is *solving* it, and that is where things get interesting.

In this notebook we will look a popular method for optimization, gradient descent, and we will see why it is sensitive to initialization. That sensitivity is the central headache that convex optimization tries to fix.

## A historical motivator: Snakes

One of the oldest energy minimization problems in our field is the *active contour model*, or Snakes model, by Kass, Witkin, Terzopoulos (1988). A snake is a curve $v(s) = (x(s), y(s))$ for $s \in [0, 1]$ that we want to wrap around an image feature. We pose the problem as
$$ \min_v \mathcal{E}_\text{snake}(v), $$
where the energy combines a smoothness term, an image-fit term, and sometimes external forces. The traditional algorithm to solve this model is gradient descent on $\mathcal{E}_\text{snake}$. It works beautifully when we initialize the curve close to the target. It struggles, often badly, when we do not.

We will not implement the Snakes model today. Instead we will reproduce the same initialization issue on a 2D test function so we can see what happens.

## A simple 2D example

Consider the function
$$ f(x, y) = (x^2 + y^2 - 1)^2 + 0.3 \cdot (x - 0.8)^2 + 0.1 \cdot \sin(4 x) \cos(4 y). $$
It has multiple local minima located near the unit circle contour. We can plot it and overlay the trajectory of gradient descent from a chosen starting point.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))
sys.path.append(str(pathlib.Path('../render_utility').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from pyplot_utils import plot_2d_function

def f(x, y):
    return (x**2 + y**2 - 1)**2 + 0.3 * (x - 0.8)**2 + 0.1 * np.sin(4 * x) * np.cos(4 * y)

def grad_f(x, y):
    common = 4 * (x**2 + y**2 - 1)
    fx = common * x + 0.6 * (x - 0.8) + 0.4 * np.cos(4 * x) * np.cos(4 * y)
    fy = common * y - 0.4 * np.sin(4 * x) * np.sin(4 * y)
    return np.array([fx, fy])

Now, we do a simple gradient descent loop. We pick a starting point, follow the negative gradient with a fixed step size, and record the trajectory.

In [ ]:
def gradient_descent(start, step=0.05, n_iters=200):
    x = np.array(start, dtype=float)
    trajectory = [x.copy()]
    for _ in range(n_iters):
        x = x - step * grad_f(*x)
        trajectory.append(x.copy())
    return np.array(trajectory)

Let's run it from three different starting points and see where each trajectory ends up.

In [ ]:
starts = [(-1.5,  1.2), (1.4, -1.0), (0.0, 1.6)]
trajectories = [gradient_descent(s) for s in starts]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, traj, start in zip(axes, trajectories, starts):
    plt.sca(ax)
    plot_2d_function(f, trajectory=traj)
    final = traj[-1]
    ax.set_title(f'Initialization at ({start[0]:.2f}, {start[1]:.2f}), '
                  f'\nSolution is ({final[0]:.2f}, {final[1]:.2f})')
plt.tight_layout()
plt.show()

Three different starts, three different ending points. None of them is wrong in the sense that all three are local minima. But only one of them is the *global* minimum, and we have no way of knowing which from inside the algorithm. This is the failure mode the introduction of the tutorial was warning about.

## What convexity buys us?

If we knew the energy were *convex*, gradient descent on it would land at the same point regardless of initialization, and that point would be the global minimum.

A number of problems in computer graphics are not naturally convex. The art of *convex relaxation* is to replace a non-convex problem with a convex one whose answer is either equal to the original (when we get lucky!) or close enough to it, in exchange for the guarantees that convexity gives us.

## Summary

In this notebook, we saw that even a simple 2D non-convex function is enough to make a generic solver depend critically on initialization. In the next notebook we'll define convexity more carefully and look at what convex sets and convex functions look like.

## References

[1] *Snakes: Active contour models*. Kass, M.; Witkin, A.; Terzopoulos, D.
       International Journal of Computer Vision 1 (4): 321 (1988).
       :DOI:`10.1007/BF00133570`